# Kafaat — Elderly Care Data Analysis

This notebook is the reproducible analysis file for the submission.

**Workflow:** Load validated data → verify schema → data-quality checks → descriptive analysis → visualizations → management findings.

> Important: the analysis uses only columns that already exist in the submitted dataset. It does not create new source-data columns.


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data_clean" / "FinalOne.xlsx"
FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)


## 2. Load the cleaned dataset

In [ ]:
df = pd.read_excel(DATA_PATH)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Column names:")
print(df.columns.tolist())

df.head()


## 3. Schema validation

In [ ]:
EXPECTED_COLUMNS = [
    "السنة", "الجنس", "الجنسية", "العمر", "الحالة الاجتماعية",
    "المستوى التعليمي", "الخدمة", "أسباب الدخول",
    "المنطقة الإدارية", "الفرع", "نوع الخدمة"
]

assert df.columns.tolist() == EXPECTED_COLUMNS, "Dataset schema differs from the expected submitted structure."
print("Schema check passed.")


## 4. Data quality checks

The checks below do not automatically delete ambiguous records. Duplicates and statistical outliers are reported for review.


In [ ]:
missing_by_column = df.isna().sum()
missing_total = int(missing_by_column.sum())

print("Total missing cells:", missing_total)
display(missing_by_column.to_frame("missing_values"))


In [ ]:
duplicate_excess = int(df.duplicated(keep="first").sum())
print("Fully duplicated excess records:", duplicate_excess)
print("Duplicate percentage:", round(duplicate_excess / len(df) * 100, 2), "%")


In [ ]:
age = pd.to_numeric(df["العمر"], errors="coerce")
q1 = age.quantile(0.25)
q3 = age.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

age_outliers = df[(age < lower_bound) | (age > upper_bound)]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential age outliers:", len(age_outliers))

age_outliers


In [ ]:
placeholders = ["غير محدد", "غير مسجل", "Unknown", "Not Specified", "Not Recorded"]

placeholder_counts = {
    column: int(df[column].astype("string").isin(placeholders).sum())
    for column in df.columns
}

pd.Series(placeholder_counts, name="placeholder_count").sort_values(ascending=False)


## 5. Descriptive analysis

In [ ]:
records_by_period = df["السنة"].astype(str).value_counts().sort_index()
records_by_period


In [ ]:
gender_distribution = pd.crosstab(
    df["السنة"].astype(str),
    df["الجنس"],
    normalize="index"
).mul(100).round(1)

gender_distribution


In [ ]:
age_summary = df.groupby("السنة")["العمر"].agg(["count", "mean", "min", "max"]).round(1)
age_summary


In [ ]:
nationality_distribution = df["الجنسية"].value_counts()
nationality_distribution


In [ ]:
top_admission_reasons = df["أسباب الدخول"].value_counts().head(10)
top_admission_reasons


In [ ]:
top_regions = df["المنطقة الإدارية"].value_counts().head(10)
top_regions


In [ ]:
service_type_distribution = df["نوع الخدمة"].value_counts()
service_type_distribution


## 6. Visualizations

In [ ]:
ax = records_by_period.plot(kind="bar", figsize=(8, 4.5))
ax.set_title("Records by Reporting Period")
ax.set_xlabel("Reporting Period")
ax.set_ylabel("Records")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "records_by_period.png", dpi=160)
plt.show()


In [ ]:
ax = top_regions.head(8).sort_values().plot(kind="barh", figsize=(8, 4.5))
ax.set_title("Top Administrative Regions")
ax.set_xlabel("Records")
ax.set_ylabel("Region")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_regions.png", dpi=160)
plt.show()


## 7. Executive findings

In [ ]:
total_records = len(df)
avg_age = pd.to_numeric(df["العمر"], errors="coerce").mean()
largest_region = df["المنطقة الإدارية"].value_counts().idxmax()
largest_region_count = df["المنطقة الإدارية"].value_counts().max()
top_reason = df["أسباب الدخول"].value_counts().idxmax()
top_reason_count = df["أسباب الدخول"].value_counts().max()

print(f"Total records: {total_records:,}")
print(f"Average age: {avg_age:.1f}")
print(f"Largest region: {largest_region} ({largest_region_count:,}; {largest_region_count/total_records:.1%})")
print(f"Most common admission reason: {top_reason} ({top_reason_count:,}; {top_reason_count/total_records:.1%})")
print(f"Missing cells: {missing_total:,}")
print(f"Fully duplicated excess records requiring validation: {duplicate_excess:,}")
print(f"Potential age outliers requiring review: {len(age_outliers):,}")


## 8. Responsible analysis note

- Statistical outliers are **flagged**, not automatically removed.
- Fully duplicated records are **reported**, not automatically deleted, because the submitted data does not contain a unique case identifier that proves they are duplicate beneficiaries.
- AI may assist with presentation and wording, but all reported numbers must be traceable to calculations in this notebook.
- Human review remains required for ambiguous data-quality cases.
